# 02 — Text Preprocessing

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Apply the complete preprocessing pipeline to prepare the dataset for TF-IDF feature extraction.

**Pipeline Order** (per TRD):

```
Raw Text → Case Folding → Text Cleaning → Tokenization → Stopword Removal → Stemming → Join Tokens
```

---

## 1. Setup

In [1]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import (
    DATASET_PATH,
    CLEAN_DATASET_PATH,
    PREPROCESSING_REPORTS_DIR,
)
from src.config.preprocessing_config import PreprocessingConfig
from src.data.load_dataset import load_dataset
from src.preprocessing.validator import validate_dataset
from src.preprocessing.pipeline import (
    preprocess_dataframe,
    get_example_transformations,
)
from src.preprocessing.report import (
    generate_preprocessing_report,
    export_preprocessing_statistics,
    save_report,
)
from src.utils.save_dataset import save_dataset

print('Setup complete.')

Setup complete.


---
## 2. Load Dataset

In [2]:
df = load_dataset()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

✓ Dataset loaded successfully: 41,556 rows, 2 columns
Shape: (41556, 2)
Columns: ['text', 'label']


,text,label
0,setiap orang adalah seorang gadis yang akan me...,harassment
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,harassment
2,karena beberapa orang tidak ada yang lebih bai...,harassment
3,bro aku pelatih jv tahun lalu di skyline dan a...,harassment
4,wanitawanita ini benarbenar mengingatkan saya ...,harassment


---
## 3. Validate Dataset

In [3]:
validation = validate_dataset(df)
print(validation['summary'])

Dataset Validation Report (41,556 rows)
--------------------------------------------------
          missing_text: ✓
            empty_text: ✓
         missing_label: ✓
        invalid_labels: ✓
        duplicate_rows: ✓
        duplicate_text: ✓
--------------------------------------------------
                Status: PASS


---
## 4. Configure Preprocessing Pipeline

In [4]:
# All steps enabled by default (per TRD pipeline specification)
# To disable a step, set its flag to False
config = PreprocessingConfig(
    enable_case_folding=True,
    enable_url_removal=True,
    enable_html_removal=True,
    enable_mention_removal=True,
    enable_hashtag_removal=True,
    enable_emoji_removal=True,
    enable_punctuation_removal=True,
    enable_number_removal=True,
    enable_stopword_removal=True,
    enable_stemming=True,
    enable_slang_normalization=False,  # Optional, disabled by default
)

print('Active preprocessing steps:')
for i, step in enumerate(config.get_active_steps(), 1):
    print(f'  {i}. {step}')

Active preprocessing steps:
  1. Case Folding
  2. URL Removal
  3. HTML Removal
  4. Mention Removal
  5. Hashtag Removal
  6. Emoji Removal
  7. Punctuation Removal
  8. Number Removal
  9. Stopword Removal
  10. Stemming


---
## 5. Run Preprocessing Pipeline

⚠️ **Note**: Stemming with Sastrawi on 41K+ rows may take 5–15 minutes.

In [5]:
def progress(current: int, total: int) -> None:
    """Print progress updates."""
    pct = current / total * 100
    print(f'  Processing: {current:,}/{total:,} ({pct:.1f}%)', end='\r')

print('Starting preprocessing pipeline...')
print(f'Config: {len(config.get_active_steps())} active steps\n')

df_processed, stats = preprocess_dataframe(
    df=df,
    config=config,
    text_col='text',
    output_col='text_clean',
    progress_callback=progress,
)

print(f'\n\n✓ Preprocessing complete!')
print(f'  Time: {stats["processing_time_seconds"]:.2f}s')
print(f'  Speed: {stats["rows_per_second"]:.0f} rows/sec')
print(f'  Avg words before: {stats["avg_word_count_before"]}')
print(f'  Avg words after:  {stats["avg_word_count_after"]}')

Starting preprocessing pipeline...
Config: 10 active steps

  Processing: 41,000/41,556 (98.7%)

✓ Preprocessing complete!
  Time: 3956.15s
  Speed: 10 rows/sec
  Avg words before: 34.47
  Avg words after:  21.47


---
## 6. Inspect Results

In [6]:
# Show before/after comparison
print('Sample transformations:\n')
for i, row in df_processed.head(5).iterrows():
    print(f'[{row["label"]}]')
    print(f'  Before: {row["text"][:100]}...')
    print(f'  After:  {row["text_clean"][:100]}...')
    print()

Sample transformations:

[harassment]
  Before: setiap orang adalah seorang gadis yang akan mengganggu saya di sekolah tinggi...
  After:  orang gadis ganggu sekolah...

[harassment]
  Before: bahwa pos ab kpop stans pergi ke sekolah bersamasama dan semua orang mengatakan mereka akan menggert...
  After:  pos ab kpop stans pergi sekolah bersamasama orang gertak tentara takut kpop stan...

[harassment]
  Before: karena beberapa orang tidak ada yang lebih baik untuk dilakukan atau mereka adalah pengganggu di sek...
  After:  orang ganggu sekolah outgrew...

[harassment]
  Before: bro aku pelatih jv tahun lalu di skyline dan aku harus mengubah air menjadi anggur kami memenangkan ...
  After:  bro latih jv skyline ubah air anggur menang tanding tim super atletik gertak kalah tim ahli lulus am...

[harassment]
  Before: wanitawanita ini benarbenar mengingatkan saya pada anak ayam sma dengan semua penindasan oleh satu d...
  After:  wanitawanita benarbenar anak ayam sma tindas diva orang men

In [ ]:
# Check for empty results after preprocessing
empty_count = (df_processed['text_clean'].astype(str).str.strip() == '').sum()
print(f'Empty texts after preprocessing: {empty_count:,}')
print(f'Total rows: {len(df_processed):,}')

---
## 7. Save Processed Dataset

In [7]:
# Save with text, label, and text_clean columns
save_dataset(
    df=df_processed[['text', 'label', 'text_clean']],
    filepath=CLEAN_DATASET_PATH,
    overwrite=True,
)

print(f'\nSaved columns: {list(df_processed[["text", "label", "text_clean"]].columns)}')

✓ Dataset saved: /home/zapp/Kampus/PM/dataset/processed/final_dataset_clean.csv (41,556 rows)

Saved columns: ['text', 'label', 'text_clean']


---
## 8. Generate Preprocessing Report

In [8]:
# Get example transformations
examples = get_example_transformations(
    df_processed, text_col='text', output_col='text_clean', n=5,
)

# Generate markdown report
report_content = generate_preprocessing_report(
    total_rows=stats['total_rows'],
    processed_rows=stats['processed_rows'],
    removed_empty=stats.get('empty_after_preprocessing', 0),
    removed_duplicates=0,  # No rows removed, only transformed
    avg_length_before=stats['avg_word_count_before'],
    avg_length_after=stats['avg_word_count_after'],
    examples=examples,
    processing_time=stats['processing_time_seconds'],
    config=config,
)

# Save report
report_path = os.path.join(PREPROCESSING_REPORTS_DIR, 'preprocessing_summary.md')
save_report(report_content, report_path)
print(f'✓ Report saved: {report_path}')

# Export statistics CSV
stats_path = os.path.join(PREPROCESSING_REPORTS_DIR, 'preprocessing_statistics.csv')
export_preprocessing_statistics(stats, stats_path)
print(f'✓ Statistics saved: {stats_path}')

✓ Report saved: /home/zapp/Kampus/PM/reports/preprocessing/preprocessing_summary.md
✓ Statistics saved: /home/zapp/Kampus/PM/reports/preprocessing/preprocessing_statistics.csv


---
## Summary

Preprocessing is complete. Generated outputs:

- `dataset/processed/final_dataset_clean.csv` — Preprocessed dataset
- `reports/preprocessing/preprocessing_summary.md` — Report
- `reports/preprocessing/preprocessing_statistics.csv` — Statistics

**Next Step**: `03_feature_engineering.ipynb` — TF-IDF Feature Extraction